# Real atmospheric and ocean forcing

**Learning goals:** Inspect real ERA5 and HYCOM data and connect them to SCHISM forcing objects.

**Prerequisites:** Lesson 3; access to the shared `rom-py/rompy-test-data` fixture bundle.

**Execution contract:** This lesson is **configuration-only**. Documentation rendering never executes SCHISM, downloads data, or requires MPI/Docker.

## Checkpoint

By the end of this lesson, record what was configured and which steps still require a model runtime.

Previous: [journey_03_schism_grid_data](../journey_03_schism_grid_data/)

Next: [journey_05_schism_boundaries](../journey_05_schism_boundaries/)


## Why this matters: SCHISM data preparation

**Without Rompy:** preparing HYCOM boundary conditions can mean downloading a large global dataset, selecting the run period and region, interpolating onto open-boundary nodes, extracting the required variables, and writing `elev2D.th.nc` or other SCHISM files. ERA5 and tidal inputs require similarly separate preparation steps.

**With Rompy:** source objects, grid metadata, time ranges, filters, and boundary mappings are assembled into `SCHISMConfig`. Workspace generation carries out the configured cropping, interpolation, boundary extraction, and SCHISM-format conversion. The modeller still chooses appropriate datasets, variables, coordinates, numerical settings, and scientific validation checks.

The following cells show the source fields, model domain, and generated artefacts so this automation remains inspectable.


In [ ]:
import sys
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "scripts" / "schism_case_data.py").is_file())
sys.path.insert(0, str(root))
from scripts.schism_case_data import ensure_schism_data

case = ensure_schism_data()
print("Fixture directory:", case)


In [ ]:
import xarray as xr

# `case` is supplied by the previous lesson when running sequentially.
era5 = xr.open_dataset(case / "era5.nc")
hycom = xr.open_dataset(case / "hycom.nc")
print("ERA5 variables:", list(era5.data_vars))
print("HYCOM variables:", list(hycom.data_vars))
era5.close()
hycom.close()


## Visual verification: source fields

These are real cropped ERA5 and HYCOM fixtures. Plotting them before generation checks variable names, coordinate orientation, time coverage, and the spatial relationship to the regional mesh.


In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

era5 = xr.open_dataset(case / "era5.nc")
hycom = xr.open_dataset(case / "hycom.nc")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
era5.u10.isel(time=0).plot(ax=axes[0], cmap="coolwarm")
axes[0].set_title("ERA5 eastward wind")
hycom.surf_el.isel(time=0).plot(ax=axes[1], cmap="BrBG")
axes[1].set_title("HYCOM sea-surface elevation")
plt.show()
era5.close(); hycom.close()


## Atmospheric coverage and derived wind magnitude

ERA5 supplies more than one field. Here `u10` and `v10` are 10-m wind components and `msl` is mean sea-level pressure. Inspecting the extent and time axis before configuring Sflux catches reversed latitude axes, missing forecast hours, and unit assumptions early.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
era5 = xr.open_dataset(case / "era5.nc")
wind_speed = np.hypot(era5.u10, era5.v10)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
wind_speed.isel(time=0).plot(ax=axes[0], cmap="magma")
axes[0].quiver(era5.longitude, era5.latitude, era5.u10.isel(time=0), era5.v10.isel(time=0), color="white", scale=250)
axes[0].set_title("ERA5 10-m wind speed and vectors")
era5.msl.isel(time=0).plot(ax=axes[1], cmap="viridis")
axes[1].set_title("ERA5 mean sea-level pressure")
wind_speed.mean(dim=("latitude", "longitude")).plot(ax=axes[2])
axes[2].set_title("Wind time coverage")
axes[2].set_ylabel("speed (source units)")
plt.show()
print("ERA5 extent:", float(era5.longitude.min()), float(era5.longitude.max()), float(era5.latitude.min()), float(era5.latitude.max()))
print("ERA5 time range:", str(era5.time.min().values), "to", str(era5.time.max().values))
era5.close()


### Data-engineering and scientific checks

The source-to-Sflux transformation must preserve the intended time coverage and pressure/wind variables. Structural checks confirm dimensions and names only; they do not validate ERA5 provenance, unit conversion, land masking, or whether the atmospheric resolution is appropriate for this mesh.


## Rompy processing: ERA5 to SCHISM Sflux

The source plot above is only the input. Now Rompy performs the useful part: it selects the requested time window, applies the model-grid spatial context, maps `u10`, `v10`, and `msl` to SCHISM Sflux names, and writes a model-ready NetCDF file. The temporary directory keeps generated artefacts out of the repository.


In [ ]:
from tempfile import TemporaryDirectory
from rompy.core.data import DataBlob
from rompy.core.source import SourceFile
from rompy.core.filters import Filter
from rompy.core.time import TimeRange
from rompy_schism import SCHISMGrid
from rompy_schism.data import SCHISMDataSflux, SfluxAir

processing_grid = SCHISMGrid(
    hgrid=DataBlob(source=case / "hgrid.gr3"),
    vgrid=DataBlob(source=case / "vgrid.in"),
    drag=1,
)
atmos = SCHISMDataSflux(air_1=SfluxAir(
    source=SourceFile(uri=case / "era5.nc"),
    filter=Filter(sort={"coords": ["latitude"]}),
    uwind_name="u10", vwind_name="v10", prmsl_name="msl", buffer=2,
))
period = TimeRange(start="2023-01-01", end="2023-01-02", dt=3600)

with TemporaryDirectory() as output:
    generated = atmos.get(output, grid=processing_grid, time=period)
    sflux_file = next(Path(output).glob("sflux/air_*.nc"))
    sflux = xr.open_dataset(sflux_file)
    print("Rompy returned:", generated)
    print("Generated Sflux:", sflux_file.name)
    print("Generated dimensions:", dict(sflux.sizes))
    print("Generated variables:", list(sflux.data_vars))
    assert {"u10", "v10", "msl"}.issubset(sflux.data_vars)
    # Sflux is a regular source grid, not node-ordered boundary data.
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(list(sflux.sizes), list(sflux.sizes.values()), color="steelblue")
    ax.set_title("Rompy-generated Sflux dimensions")
    ax.set_ylabel("size")
    plt.show()
    sflux.close()
